# 06 — Dashboard Export

Final pipeline run: rebuild marts from clean parquets, export app-layer CSVs for Tableau,
run QA quality gates, and log a summary of what was produced.

In [ ]:
import sys

sys.path.insert(0, "..")


import pandas as pd

from src.config import DATA_APP
from src.marts.builder import build_all_marts
from src.marts.export import export_all
from src.qa.summary import build_qa_summary, evaluate_quality_gate

## 1. Rebuild Marts from Clean Parquets

In [ ]:
marts = build_all_marts()

for name, df in marts.items():
    print(f"{name}: {len(df):,} rows, {df.columns.tolist()}")

## 2. Export App-Layer CSVs for Tableau

In [ ]:
app_tables = export_all()

for name, df in app_tables.items():
    print(f"{name}: {len(df):,} rows")

## 3. Quality Gate Check

In [ ]:
qa_summary = build_qa_summary()
gate_passed = evaluate_quality_gate(qa_summary)

print(f"\nQuality gate: {'PASSED' if gate_passed else 'FAILED'}")
qa_summary

## 4. Spot-Check: Top Panels by Event Volume

In [ ]:
overview = pd.read_csv(DATA_APP / "app_overview.csv")
top_panels = overview.groupby("review_panel")["event_count_dedup"].sum().sort_values(ascending=False).head(10)
print("Top 10 panels by total deduplicated event count:")
top_panels

## 5. Output Inventory

In [ ]:
print("App CSVs ready for Tableau import:")
for f in sorted(DATA_APP.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.0f} KB")